# Browser agent (multi-turn) — Browserless × LlamaIndex (Python)

Multi-turn `browserless_agent` workflow: navigate → snapshot → click → re-snapshot → extract, all in one browser.

## How the browser persists between calls

Each `browserless_agent` call is one-shot by default: the browser closes when the call finishes. To keep it:

1. Pass `keepSessionAlive: true` on the first call.
2. Pass the `sessionId` from its reply on every later call.
3. Send `close` when you are done. An open browser holds one of your account's concurrent sessions until it idles out.

The `sessionId` identifies the browser on its own, so it works with the stock `McpToolSpec` tools even though `BasicMCPClient` opens a new MCP connection for every tool call.

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

In [ ]:
import os

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec


class BrowserlessToolSpec(McpToolSpec):
    # Pydantic rejects field names with a leading underscore, so skip
    # optional fields such as `_prompt` when building each tool's schema.
    def create_model_from_json_schema(self, schema, model_name="DynamicModel"):
        properties = {
            name: prop
            for name, prop in schema.get("properties", {}).items()
            if not name.startswith("_")
        }
        return super().create_model_from_json_schema(
            {**schema, "properties": properties}, model_name
        )


client = BasicMCPClient(
    os.environ.get("BROWSERLESS_MCP_URL", "https://mcp.browserless.io/mcp"),
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)


tools = await BrowserlessToolSpec(
    client=client, allowed_tools=["browserless_agent", "browserless_skill"]
).to_tool_list_async()
browser = next(t for t in tools if t.metadata.name == "browserless_agent")

## Direct multi-turn proof

Three `browserless_agent` calls: `goto` (kept alive) → `evaluate` → `close`. The `evaluate` call must return `"Example Domain"`, proving the navigation from call 1 persisted.

In [ ]:
def text(output):
    return "\n".join(c.text for c in output.raw_output.content if c.type == "text")


import re

first = await browser.acall(
    method="goto", params={"url": "https://example.com"}, keepSessionAlive=True
)
session_id = re.search(r"sessionId: (\S+)", text(first)).group(1)

title = text(
    await browser.acall(
        method="evaluate", params={"content": "document.title"}, sessionId=session_id
    )
)
print(title.split("\n\nsessionId")[0])
assert "Example Domain" in title, "Session state was not preserved between calls"

await browser.acall(method="close", sessionId=session_id)

## Full agent loop (LLM-driven)

The `browserless_agent` tool description tells the model to set `keepSessionAlive`, pass the `sessionId` back, and close the browser at the end, so the stock tools work in a `FunctionAgent` with no wrapper. `browserless_skill` lets the model load a tuned recipe when one exists for the target site.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

agent = FunctionAgent(
    name="BrowserlessBrowserAgent",
    description="Drives a real browser to navigate, click, and extract content.",
    llm=Anthropic(model="claude-sonnet-5-5"),
    tools=tools,
    system_prompt="You drive a real browser via the browserless_agent tool.",
)

response = await agent.run(
    "Navigate to https://news.ycombinator.com, click the first story link, and report the title and first paragraph of that page."
)
print(response)